In [27]:
import pandas
import numpy as np


In [28]:
df = pandas.read_parquet("../data/sample/parquet_delay_and_weather_24/")
label_df = pandas.read_parquet("../data/sample/parquet_delay_and_weather_24/")

In [29]:
label_df["DepDel15"]

0        0.0
1        0.0
2        0.0
3        0.0
4        1.0
        ... 
35189    1.0
35190    0.0
35191    1.0
35192    0.0
35193    1.0
Name: DepDel15, Length: 35194, dtype: float64

In [30]:
df.shape

(35194, 68)

In [31]:
df.columns

Index(['DestAirportID', 'OriginAirportID', 'Year', 'Month', 'DayofMonth',
       'DayOfWeek', 'FlightDate', 'Marketing_Airline_Network',
       'DOT_ID_Marketing_Airline', 'Operating_Airline ',
       'DOT_ID_Operating_Airline', 'Flight_Number_Operating_Airline',
       'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName',
       'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName',
       'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15',
       'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff',
       'WheelsOn', 'TaxiIn', 'CRSArrTime', 'ArrTime', 'ArrDelay',
       'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups', 'ArrTimeBlk',
       'Cancelled', 'CancellationCode', 'Diverted', 'CarrierDelay',
       'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay',
       'OriginICAO', 'OriginTimezone', 'DestICAO', 'DestTimezone',
       'CRSDepTimestamp', 'CRSArrTimestamp', 'OriginWindDirection',
       'OriginWindSpeed', 

In [32]:
missing_values = df.isnull().sum()
columns_with_nulls = missing_values[missing_values > 0]

if not columns_with_nulls.empty:
    print("Columns with null values and their counts:")
    display(columns_with_nulls)
else:
    print("No null values found in any column.")

Columns with null values and their counts:


DepTime                   478
DepDelay                  478
DepDelayMinutes           478
DepDel15                  478
DepartureDelayGroups      478
TaxiOut                   496
WheelsOff                 496
WheelsOn                  499
TaxiIn                    499
ArrTime                   499
ArrDelay                  570
ArrDelayMinutes           570
ArrDel15                  570
ArrivalDelayGroups        570
CancellationCode        34697
CarrierDelay            25962
WeatherDelay            25962
NASDelay                25962
SecurityDelay           25962
LateAircraftDelay       25962
OriginWindDirection      2269
OriginWindSpeed            14
OriginWindGusts         30920
OriginTemperature       35194
OriginDewPoint          35194
DestWindDirection        1308
DestWindSpeed              26
DestWindGusts           29388
DestVisibility             22
DestPrecipitation          13
DestClouds                 13
DestTemperature         35194
DestDewPoint            35194
dtype: int

In [33]:
cols_to_drop = [
    # Actual departure/arrival values → leakage
    "DepTime", "DepDelay", "DepDelayMinutes",
    "DepartureDelayGroups", "TaxiOut", "WheelsOff", "WheelsOn",
    "TaxiIn", "ArrTime", "ArrDelay", "ArrDelayMinutes",
    "ArrDel15", "ArrivalDelayGroups",

    # After-fact outcomes
    "Cancelled", "Diverted",

    # Redundant ID fields
    "DestAirportID", "OriginAirportID", "DOT_ID_Marketing_Airline",
    "DOT_ID_Operating_Airline", "OriginAirportSeqID",
    "DestAirportSeqID", "OriginCityMarketID", "DestCityMarketID",
    "Flight_Number_Operating_Airline", "OriginICAO", "DestICAO",

    # Duplicated / unnecessary text fields
    "OriginCityName", "DestCityName",
    "DepTimeBlk", "ArrTimeBlk",
    "OriginTimezone", "DestTimezone",

    # Redundant timestamps
    "CRSDepTimestamp", "CRSArrTimestamp",

    # Columsn that have a lot of nan that seem useful
    'CancellationCode',
    'CarrierDelay','WeatherDelay','NASDelay',"SecurityDelay","LateAircraftDelay",
    "OriginWindGusts", "OriginTemperature", "OriginDewPoint",
    "DestWindGusts",'DestTemperature','DestDewPoint'
]

label_cols_to_drop = [
    # Actual departure/arrival values → leakage
    "DepTime",
    "DepartureDelayGroups", "TaxiOut", "WheelsOff", "WheelsOn",
    "TaxiIn", "ArrTime", "ArrivalDelayGroups",

    # After-fact outcomes
    "Cancelled", "Diverted",

    # Redundant ID fields
    "DestAirportID", "OriginAirportID", "DOT_ID_Marketing_Airline",
    "DOT_ID_Operating_Airline", "OriginAirportSeqID",
    "DestAirportSeqID", "OriginCityMarketID", "DestCityMarketID",
    "Flight_Number_Operating_Airline", "OriginICAO", "DestICAO",

    # Duplicated / unnecessary text fields
    "OriginCityName", "DestCityName",
    "DepTimeBlk", "ArrTimeBlk",
    "OriginTimezone", "DestTimezone",

    # Redundant timestamps
    "CRSDepTimestamp", "CRSArrTimestamp",

    # Columsn that have a lot of nan that seem useful
    'CancellationCode',
    'CarrierDelay','WeatherDelay','NASDelay',"SecurityDelay","LateAircraftDelay",
    "OriginWindGusts", "OriginTemperature", "OriginDewPoint",
    "DestWindGusts",'DestTemperature','DestDewPoint'
]

In [34]:
from numpy import column_stack
df.drop(columns=cols_to_drop, inplace=True)

In [35]:
from numpy import column_stack
label_df.drop(columns=label_cols_to_drop, inplace=True)

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35194 entries, 0 to 35193
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Year                       35194 non-null  int32  
 1   Month                      35194 non-null  int32  
 2   DayofMonth                 35194 non-null  int32  
 3   DayOfWeek                  35194 non-null  int32  
 4   FlightDate                 35194 non-null  object 
 5   Marketing_Airline_Network  35194 non-null  object 
 6   Operating_Airline          35194 non-null  object 
 7   Origin                     35194 non-null  object 
 8   Dest                       35194 non-null  object 
 9   CRSDepTime                 35194 non-null  int32  
 10  DepDel15                   34716 non-null  float64
 11  CRSArrTime                 35194 non-null  int32  
 12  OriginWindDirection        32925 non-null  float64
 13  OriginWindSpeed            35180 non-null  flo

In [37]:
# TESTING PURPOSES
# Get the names of the first 10 columns (0 to 9) 3 and 6 onwards
#columns_to_drop_by_position = label_df.columns[6:]

# Drop these columns by name
#label_df.drop(columns=columns_to_drop_by_position, inplace=True)



In [38]:
label_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35194 entries, 0 to 35193
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Year                       35194 non-null  int32  
 1   Month                      35194 non-null  int32  
 2   DayofMonth                 35194 non-null  int32  
 3   DayOfWeek                  35194 non-null  int32  
 4   FlightDate                 35194 non-null  object 
 5   Marketing_Airline_Network  35194 non-null  object 
 6   Operating_Airline          35194 non-null  object 
 7   Origin                     35194 non-null  object 
 8   Dest                       35194 non-null  object 
 9   CRSDepTime                 35194 non-null  int32  
 10  DepDelay                   34716 non-null  float64
 11  DepDelayMinutes            34716 non-null  float64
 12  DepDel15                   34716 non-null  float64
 13  CRSArrTime                 35194 non-null  int

## Pre-Processing


#### Processing the Nan rows

In [39]:
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31316 entries, 0 to 35193
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Year                       31316 non-null  int32  
 1   Month                      31316 non-null  int32  
 2   DayofMonth                 31316 non-null  int32  
 3   DayOfWeek                  31316 non-null  int32  
 4   FlightDate                 31316 non-null  object 
 5   Marketing_Airline_Network  31316 non-null  object 
 6   Operating_Airline          31316 non-null  object 
 7   Origin                     31316 non-null  object 
 8   Dest                       31316 non-null  object 
 9   CRSDepTime                 31316 non-null  int32  
 10  DepDel15                   31316 non-null  float64
 11  CRSArrTime                 31316 non-null  int32  
 12  OriginWindDirection        31316 non-null  float64
 13  OriginWindSpeed            31316 non-null  float64


In [40]:
#defining y

label_col = "DepDel15"

y = df[label_col].astype(int)
df["label"] = y
df = df.drop(columns=[label_col])

In [41]:
## Change the CRs dep tiome and arr time to
def hhmm_to_hour(t):
    t = int(t)
    return t // 100

df["DepHour"] = df["CRSDepTime"].apply(hhmm_to_hour)
df["ArrHour"] = df["CRSArrTime"].apply(hhmm_to_hour)

def hhmm_to_minute(t):
    t = int(t)
    return t % 100

df["DepMinute"] = df["CRSDepTime"].apply(hhmm_to_minute)
df["ArrMinute"] = df["CRSArrTime"].apply(hhmm_to_minute)


In [42]:
df = df.drop(columns=["CRSDepTime", "CRSArrTime"])


In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31316 entries, 0 to 35193
Data columns (total 24 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Year                       31316 non-null  int32  
 1   Month                      31316 non-null  int32  
 2   DayofMonth                 31316 non-null  int32  
 3   DayOfWeek                  31316 non-null  int32  
 4   FlightDate                 31316 non-null  object 
 5   Marketing_Airline_Network  31316 non-null  object 
 6   Operating_Airline          31316 non-null  object 
 7   Origin                     31316 non-null  object 
 8   Dest                       31316 non-null  object 
 9   OriginWindDirection        31316 non-null  float64
 10  OriginWindSpeed            31316 non-null  float64
 11  OriginVisibility           31316 non-null  float64
 12  OriginPrecipitation        31316 non-null  object 
 13  OriginClouds               31316 non-null  object 


In [44]:
df["FlightDate"] = pandas.to_datetime(df["FlightDate"])
df["is_weekend"] = df["FlightDate"].dt.dayofweek.isin([5, 6]).astype(int)

In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31316 entries, 0 to 35193
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Year                       31316 non-null  int32         
 1   Month                      31316 non-null  int32         
 2   DayofMonth                 31316 non-null  int32         
 3   DayOfWeek                  31316 non-null  int32         
 4   FlightDate                 31316 non-null  datetime64[ns]
 5   Marketing_Airline_Network  31316 non-null  object        
 6   Operating_Airline          31316 non-null  object        
 7   Origin                     31316 non-null  object        
 8   Dest                       31316 non-null  object        
 9   OriginWindDirection        31316 non-null  float64       
 10  OriginWindSpeed            31316 non-null  float64       
 11  OriginVisibility           31316 non-null  float64       
 12  OriginPre

In [46]:
def arr_to_str(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return "None"
    if isinstance(val, (list, np.ndarray)):
        if len(val) == 0:
            return "None"
        return ",".join(map(str, val))
    return str(val)
weather_array_cols = [
    "OriginPrecipitation", "OriginClouds",
    "DestPrecipitation", "DestClouds"
]

for col in weather_array_cols:
    df[col] = df[col].apply(arr_to_str)


In [47]:
df.rename(columns={'Operating_Airline ': 'Operating_Airline'}, inplace=True)

In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31316 entries, 0 to 35193
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Year                       31316 non-null  int32         
 1   Month                      31316 non-null  int32         
 2   DayofMonth                 31316 non-null  int32         
 3   DayOfWeek                  31316 non-null  int32         
 4   FlightDate                 31316 non-null  datetime64[ns]
 5   Marketing_Airline_Network  31316 non-null  object        
 6   Operating_Airline          31316 non-null  object        
 7   Origin                     31316 non-null  object        
 8   Dest                       31316 non-null  object        
 9   OriginWindDirection        31316 non-null  float64       
 10  OriginWindSpeed            31316 non-null  float64       
 11  OriginVisibility           31316 non-null  float64       
 12  OriginPre

In [49]:
y.info()

<class 'pandas.core.series.Series'>
Index: 31316 entries, 0 to 35193
Series name: DepDel15
Non-Null Count  Dtype
--------------  -----
31316 non-null  int64
dtypes: int64(1)
memory usage: 489.3 KB


## SPARK setup


In [50]:
from pyspark.sql import SparkSession
import numpy as np

In [51]:
spark = SparkSession.builder.appName("Pandas to Spark").getOrCreate()
sdf = spark.createDataFrame(df)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/18 14:02:06 WARN Utils: Your hostname, codespaces-90a6fa, resolves to a loopback address: 127.0.0.1; using 10.0.3.118 instead (on interface eth0)
25/11/18 14:02:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/18 14:02:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [52]:
sdf.printSchema()
sdf.show()

root
 |-- Year: long (nullable = true)
 |-- Month: long (nullable = true)
 |-- DayofMonth: long (nullable = true)
 |-- DayOfWeek: long (nullable = true)
 |-- FlightDate: timestamp (nullable = true)
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- Operating_Airline: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- OriginWindDirection: double (nullable = true)
 |-- OriginWindSpeed: double (nullable = true)
 |-- OriginVisibility: double (nullable = true)
 |-- OriginPrecipitation: string (nullable = true)
 |-- OriginClouds: string (nullable = true)
 |-- DestWindDirection: double (nullable = true)
 |-- DestWindSpeed: double (nullable = true)
 |-- DestVisibility: double (nullable = true)
 |-- DestPrecipitation: string (nullable = true)
 |-- DestClouds: string (nullable = true)
 |-- label: long (nullable = true)
 |-- DepHour: long (nullable = true)
 |-- ArrHour: long (nullable = true)
 |-- DepMinute: long (nullable = true)
 |

25/11/18 14:02:22 WARN TaskSetManager: Stage 0 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.


+----+-----+----------+---------+-------------------+-------------------------+-----------------+------+----+-------------------+---------------+----------------+-------------------+--------------------+-----------------+-------------+--------------+-----------------+--------------------+-----+-------+-------+---------+---------+----------+
|Year|Month|DayofMonth|DayOfWeek|         FlightDate|Marketing_Airline_Network|Operating_Airline|Origin|Dest|OriginWindDirection|OriginWindSpeed|OriginVisibility|OriginPrecipitation|        OriginClouds|DestWindDirection|DestWindSpeed|DestVisibility|DestPrecipitation|          DestClouds|label|DepHour|ArrHour|DepMinute|ArrMinute|is_weekend|
+----+-----+----------+---------+-------------------+-------------------------+-----------------+------+----+-------------------+---------------+----------------+-------------------+--------------------+-----------------+-------------+--------------+-----------------+--------------------+-----+-------+-------+---

In [53]:
cat_cols = [
    "Marketing_Airline_Network",
    "Operating_Airline",
    "Origin",
    "Dest",
    "OriginPrecipitation",
    "OriginClouds",
    "DestPrecipitation",
    "DestClouds"
]

num_cols = [
    "Year",
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "OriginWindDirection",
    "OriginWindSpeed",
    "OriginVisibility",
    "DestWindDirection",
    "DestWindSpeed",
    "DestVisibility",
    "DepHour",
    "ArrHour",
    "DepMinute",
    "ArrMinute",
    "is_weekend"
]




In [54]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import GBTClassifier, RandomForestClassifier
from pyspark.ml import Pipeline

# StringIndexers for all categorical features
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx",
        handleInvalid="keep"
    )
    for c in cat_cols
]

# OneHotEncoders for all indexed categorical features
encoders = [
    OneHotEncoder(
        inputCol=c + "_idx",
        outputCol=c + "_ohe"
    )
    for c in cat_cols
]

# VectorAssembler that combines encoded categoricals and numeric features
assembler = VectorAssembler(
    inputCols=[c + "_ohe" for c in cat_cols] + num_cols,
    outputCol="features"
)

# Classifier THIS GIVE A MODEL WITH AROUND 75% accuracy really quickly
gbt = GBTClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=5,
    maxIter=50
)




# Pipeline
pipeline = Pipeline(stages=indexers + encoders + [assembler, gbt])

# Train test split
train_df, test_df = sdf.randomSplit([0.8, 0.2], seed=42)

# Fit model
gbt_model = pipeline.fit(train_df)

# Predictions
predictions = gbt_model.transform(test_df)



25/11/18 14:02:26 WARN TaskSetManager: Stage 1 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:02:30 WARN TaskSetManager: Stage 7 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:02:31 WARN TaskSetManager: Stage 13 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:02:33 WARN TaskSetManager: Stage 19 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:02:34 WARN TaskSetManager: Stage 25 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:02:35 WARN TaskSetManager: Stage 31 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:02:36 WARN TaskSetManager: Stage 37 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18

In [55]:
predictions.select("label", "prediction", "probability").show(50, truncate=False)

25/11/18 14:03:22 WARN TaskSetManager: Stage 553 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:03:22 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|0    |0.0       |[0.8121426218677696,0.18785737813223036]|
|0    |0.0       |[0.8139729567129751,0.1860270432870249] |
|0    |0.0       |[0.7626558253844711,0.23734417461552892]|
|0    |0.0       |[0.8083933090880201,0.1916066909119799] |
|0    |0.0       |[0.8035743504670713,0.19642564953292874]|
|0    |0.0       |[0.8382649870190363,0.1617350129809637] |
|0    |0.0       |[0.7685332783902232,0.23146672160977677]|
|0    |0.0       |[0.8948911412001447,0.10510885879985532]|
|0    |0.0       |[0.88284610507132,0.11715389492868]     |
|0    |0.0       |[0.8180358309496443,0.18196416905035573]|
|1    |0.0       |[0.8052946517738869,0.19470534822611307]|
|0    |0.0       |[0.777653469444356,0.22234653055564402] |
|0    |0.0       |[0.884579761967487,0.115420238032513]   |
|0    |0.0       |[0.8688255377803671,0.

In [56]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Accuracy
acc_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_eval.evaluate(predictions)

# F1 (overall)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)
f1 = f1_eval.evaluate(predictions)

print("Accuracy:", accuracy)
print("F1:", f1)

25/11/18 14:03:23 WARN TaskSetManager: Stage 554 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:03:24 WARN TaskSetManager: Stage 556 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.


Accuracy: 0.7553556827473425
F1: 0.7126112846857692


In [57]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=100,   # try 100 first
    maxDepth=10     # a bit smaller
)

pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])

train_df, test_df = sdf.randomSplit([0.8, 0.2], seed=42)

rf_model = pipeline.fit(train_df)

predictions = rf_model.transform(test_df)

25/11/18 14:03:25 WARN TaskSetManager: Stage 558 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:03:26 WARN TaskSetManager: Stage 564 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:03:26 WARN TaskSetManager: Stage 570 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:03:27 WARN TaskSetManager: Stage 576 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:03:28 WARN TaskSetManager: Stage 582 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:03:29 WARN TaskSetManager: Stage 588 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:03:30 WARN TaskSetManager: Stage 594 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.

In [58]:
predictions.select("label", "prediction", "probability").show(50, truncate=False)

25/11/18 14:03:54 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
25/11/18 14:03:54 WARN TaskSetManager: Stage 633 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.


+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|0    |0.0       |[0.6823928096256608,0.3176071903743391] |
|0    |0.0       |[0.769836163074881,0.2301638369251189]  |
|0    |0.0       |[0.6597213782878906,0.34027862171210943]|
|0    |0.0       |[0.7764952346049552,0.22350476539504474]|
|0    |0.0       |[0.7608175816671102,0.23918241833288975]|
|0    |0.0       |[0.8129581175297893,0.18704188247021059]|
|0    |0.0       |[0.6825367435004136,0.3174632564995863] |
|0    |0.0       |[0.8454992433053404,0.15450075669465957]|
|0    |0.0       |[0.8171068132982761,0.18289318670172391]|
|0    |0.0       |[0.7427937215454865,0.25720627845451344]|
|1    |0.0       |[0.7396806540844102,0.2603193459155898] |
|0    |0.0       |[0.6966590938612591,0.30334090613874093]|
|0    |0.0       |[0.844799006627569,0.15520099337243093] |
|0    |0.0       |[0.7798531110886181,0.

In [59]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Accuracy
acc_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_eval.evaluate(predictions)

# F1 (overall)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)
f1 = f1_eval.evaluate(predictions)

print("Accuracy:", accuracy)
print("F1:", f1)


25/11/18 14:03:54 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB
25/11/18 14:03:54 WARN TaskSetManager: Stage 634 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:03:56 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB
25/11/18 14:03:56 WARN TaskSetManager: Stage 636 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.


Accuracy: 0.740801308258381
F1: 0.6347527634000789


## Run for best model performance will take a really long time

In [60]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.classification import GBTClassifier
from pyspark.ml import Pipeline

# Re-define GBTClassifier without featuresCol, as it will be set by the pipeline
gbt_cv = GBTClassifier(labelCol="label")

# Re-create the pipeline including the GBTClassifier
pipeline_cv = Pipeline(stages=indexers + encoders + [assembler, gbt_cv])

paramGrid = (
    ParamGridBuilder()
    .addGrid(gbt_cv.maxDepth, [3, 5, 7])
    .addGrid(gbt_cv.maxIter, [30, 50, 100])
    .addGrid(gbt_cv.stepSize, [0.05, 0.1, 0.2])
    .addGrid(gbt_cv.maxBins, [32, 64])
    .build()
)

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="probability",
    metricName="areaUnderROC"
)

cv = CrossValidator(
    estimator=pipeline_cv, # Use the entire pipeline as the estimator
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=4     # use multiple cores
)

cv_model = cv.fit(train_df)
best_model = cv_model.bestModel

predictions = best_model.transform(test_df)

25/11/18 14:03:58 WARN TaskSetManager: Stage 638 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:04:00 WARN TaskSetManager: Stage 641 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:04:00 WARN TaskSetManager: Stage 639 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:04:00 WARN TaskSetManager: Stage 640 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:04:00 WARN TaskSetManager: Stage 642 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:04:00 WARN TaskSetManager: Stage 643 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:04:00 WARN TaskSetManager: Stage 644 contains a task of very large size (2377 KiB). The maximum recommended task size is 1000 KiB.

KeyboardInterrupt: 

25/11/18 14:24:53 WARN TaskSetManager: Stage 32002 contains a task of very large size (2380 KiB). The maximum recommended task size is 1000 KiB.


In [61]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.classification import GBTClassifier
from pyspark.ml import Pipeline

# GBT with label set, features come from assembler
gbt_cv = GBTClassifier(labelCol="label", featuresCol="features")

pipeline_cv = Pipeline(stages=indexers + encoders + [assembler, gbt_cv])

paramGrid = (
    ParamGridBuilder()
    .addGrid(gbt_cv.maxDepth, [3, 5])
    .addGrid(gbt_cv.maxIter, [40, 80])
    .addGrid(gbt_cv.stepSize, [0.1])    # keep step size fixed for now
    .build()
)

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="probability",
    metricName="areaUnderROC"
)

cv = CrossValidator(
    estimator=pipeline_cv,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=2,        # use 2 folds instead of 3
    parallelism=4
)

cv_model = cv.fit(train_df)
best_model = cv_model.bestModel

save_path = "./spark_gbt_model"
best_model.save(save_path)
print(f"Model successfully saved to: {save_path}")

predictions = best_model.transform(test_df)


25/11/18 14:25:02 WARN TaskSetManager: Stage 32152 contains a task of very large size (2380 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:25:02 WARN DAGScheduler: Broadcasting large task binary with size 1687.3 KiB
25/11/18 14:25:02 WARN TaskSetManager: Stage 32154 contains a task of very large size (2382 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:25:02 WARN DAGScheduler: Broadcasting large task binary with size 1590.4 KiB
25/11/18 14:25:02 WARN TaskSetManager: Stage 32156 contains a task of very large size (2382 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:25:02 WARN TaskSetManager: Stage 32158 contains a task of very large size (2380 KiB). The maximum recommended task size is 1000 KiB.
25/11/18 14:25:02 WARN DAGScheduler: Broadcasting large task binary with size 1014.0 KiB
25/11/18 14:25:02 WARN TaskSetManager: Stage 32160 contains a task of very large size (2380 KiB). The maximum recommended task size is 1000 KiB.
25/11/18

Model successfully saved to: ./spark_gbt_model


In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Accuracy
acc_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_eval.evaluate(predictions)

# F1 (overall)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)
f1 = f1_eval.evaluate(predictions)

print("Accuracy:", accuracy)
print("F1:", f1)

Accuracy: 0.7538838920686836
F1: 0.7119166555310095
